# AlexNet — ImageNet Classification with Deep Convolutional Neural Networks (2012)

_Papers · Computer Vision_

**A deep CNN with ReLU, dropout, data augmentation and GPU training crushed ImageNet and launched the deep-learning era.**

---

This notebook is a practice scaffold for the **AlexNet — ImageNet Classification with Deep Convolutional Neural Networks (2012)** lesson. We build it up one step at a time: recompute the conv-output-size formula, assemble a scaled-down AlexNet, train it on a CIFAR-10 subset, and run the ReLU-vs-tanh ablation. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the CIFAR10 dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
print("dataset: CIFAR10   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We build it in four steps: (1) recompute the conv-output-size formula on the paper's conv1, (2) define a scaled-down AlexNet for 32x32 CIFAR-10, (3) set up the data and a training/eval loop, and (4) run the ReLU-vs-tanh ablation that shows why ReLU trains faster.

### Step 1 — Recompute the conv-output-size formula

A convolution turns a $W\times W$ input into $O\times O$ where $O = \lfloor (W - K + 2P)/S \rfloor + 1$ for kernel $K$, stride $S$, padding $P$. We apply it to AlexNet's first layer. The paper's text says a 224-input gives 54, but reading Figure 2 with a 227 input gives the more commonly cited 55 — both are just this formula with different inputs.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)

# Recompute the worked example: conv output size O = floor((W - K + 2P)/S) + 1.
def conv_out(W, K, S, P):
    return (W - K + 2 * P) // S + 1

print("conv1, paper numbers (W=224, K=11, S=4, P=0):", conv_out(224, 11, 4, 0))   # -> 54
print("conv1, paper Fig.2 reading (W=227, K=11, S=4, P=0):", conv_out(227, 11, 4, 0))  # -> 55
# 54x54x96 with the clean no-pad 224 input; 55 with the 227/padded reading the figure uses.

### Step 2 — Define a scaled-down AlexNet

AlexNet stacks 5 convolutional layers (a feature extractor) followed by 3 fully-connected layers (a classifier), with ReLU activations and dropout in the FC layers (§ 4.2). We shrink it to fit 32x32 CIFAR-10 images. The `act` argument swaps the activation, which is the lever for the ReLU-vs-tanh ablation later.

In [ ]:
# A scaled-down AlexNet for 32x32 CIFAR-10: 5 conv + 3 FC, ReLU, dropout.
class TinyAlexNet(nn.Module):
    def __init__(self, n_classes=10, act="relu"):
        super().__init__()
        Act = nn.ReLU if act == "relu" else nn.Tanh   # the ablation switch
        self.features = nn.Sequential(
            nn.Conv2d(3,   32, 3, padding=1), Act(), nn.MaxPool2d(2),   # 32x32 -> 16x16
            nn.Conv2d(32,  64, 3, padding=1), Act(), nn.MaxPool2d(2),   # 16x16 -> 8x8
            nn.Conv2d(64, 128, 3, padding=1), Act(),                    # conv 3 (no pool)
            nn.Conv2d(128,128, 3, padding=1), Act(),                    # conv 4 (no pool)
            nn.Conv2d(128, 64, 3, padding=1), Act(), nn.MaxPool2d(2),   # 8x8 -> 4x4  (conv 5)
        )
        # After features: 64 channels x 4 x 4 = 1024 features.
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(64 * 4 * 4, 256), Act(),        # FC1  (dropout, §4.2)
            nn.Dropout(0.5), nn.Linear(256, 128),        Act(),        # FC2  (dropout)
            nn.Linear(128, n_classes),                                 # FC3 -> 10 logits
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

### Step 3 — Load CIFAR-10 and set up a train/eval loop

We use a small CIFAR-10 subset with light augmentation (random crop + horizontal flip, echoing § 4.1) so training is fast. The `run` function trains for a few epochs, then evaluates test accuracy with `net.eval()` — which switches dropout OFF for deterministic inference, a common pitfall if forgotten.

In [ ]:
# A CIFAR-10 subset with light augmentation (random crop + h-flip, echoing §4.1).
train_tfm = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                       T.ToTensor(),
                       T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])
test_tfm  = T.Compose([T.ToTensor(),
                       T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])

train_full = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tfm)
test_full  = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tfm)
train_set  = torch.utils.data.Subset(train_full, range(5000))   # small + fast
test_set   = torch.utils.data.Subset(test_full,  range(2000))
train_ld   = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_ld    = torch.utils.data.DataLoader(test_set,  batch_size=256)
device     = "cuda" if torch.cuda.is_available() else "cpu"


def run(act="relu", epochs=5):
    torch.manual_seed(0)
    net = TinyAlexNet(act=act).to(device)
    opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
    lf  = nn.CrossEntropyLoss()
    curve = []
    for ep in range(epochs):
        net.train(); tot = 0.0; nb = 0
        for xb, yb in train_ld:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = lf(net(xb), yb); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        curve.append(tot / nb)
        print(f"  [{act}] epoch {ep}  train loss {curve[-1]:.4f}")
    # Test accuracy (net.eval() turns dropout OFF -- §4.2 / common pitfall).
    net.eval(); correct = 0; total = 0
    with torch.no_grad():
        for xb, yb in test_ld:
            xb, yb = xb.to(device), yb.to(device)
            pred = net(xb).argmax(1)
            correct += (pred == yb).sum().item(); total += yb.size(0)
    acc = correct / total
    print(f"  [{act}] TEST ACCURACY on 2000 CIFAR-10 images: {acc:.3f}")
    return curve, acc

### Step 4 — Run the ReLU-vs-tanh ablation

We train the same architecture twice, changing only the activation. ReLU's non-saturating positive region keeps gradients flowing, so its loss should drop faster and usually end with higher test accuracy than tanh — the paper's § 3.1 claim, reproduced on a small run.

In [ ]:
print("\nTraining scaled-down AlexNet (ReLU):")
relu_curve, relu_acc = run("relu")

# Ablation: same net with tanh instead of ReLU -- ReLU should train faster (§3.1).
print("\nABLATION -- same architecture with tanh:")
tanh_curve, tanh_acc = run("tanh")

print("\nReLU train loss/epoch:", [round(c, 3) for c in relu_curve], " test acc:", round(relu_acc, 3))
print("tanh train loss/epoch:", [round(c, 3) for c in tanh_curve], " test acc:", round(tanh_acc, 3))
# ReLU's loss drops faster and usually ends with higher test accuracy.
# (Exact numbers vary by hardware/seed; this is OUR small run, NOT the paper's reported ImageNet result.)

## Visualize the data & results

_Does the ReLU net train faster than the matched tanh net (same scaled-down AlexNet, CIFAR-10 subset)?_

### Step 1 — Re-run both activations

Using the same `run` helper, we retrain the ReLU and tanh nets and keep their per-epoch loss curves. Everything (architecture, optimizer, data subset, seed) is identical except the activation, so the curves isolate its effect.

In [ ]:
# Reproduce the ReLU-vs-tanh training-speed contrast.
# (Same TinyAlexNet + CIFAR-10 subset as above; run() returns the loss curve.)
relu_curve, _ = run("relu", epochs=5)
tanh_curve, _ = run("tanh", epochs=5)

### Step 2 — Compare the loss curves epoch by epoch

We print the two losses side by side at each epoch. ReLU's curve should sit below tanh's at every epoch — the visible signature of faster training. These are our small-run numbers, not the paper's reported result.

In [ ]:
for ep in range(5):
    print(f"epoch {ep}:  ReLU {relu_curve[ep]:.3f}   tanh {tanh_curve[ep]:.3f}")
# ReLU's curve sits below tanh's at every epoch -- our small run, not the paper's reported result.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have the scaled-down AlexNet training with ReLU. Replace every ReLU with
            nn.Tanh, keep depth, width, optimizer, data, and seed identical, and retrain. What do
            you expect to see in the early training loss, and what does it demonstrate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Swap only the activation: every nn.ReLU becomes nn.Tanh; change nothing else. — _An honest ablation changes exactly one thing &mdash; the nonlinearity &mdash; so any difference is attributable to it._
- Watch the first one or two epochs: the ReLU net's training loss drops noticeably faster than the tanh net's. — _$\tanh$ saturates for large-magnitude inputs (gradient near $0$), so updates to those neurons stall; ReLU keeps a gradient of $1$ on the positive side, so learning proceeds._
- Conclude that the activation choice, not capacity, drove the speed-up. — _Both nets have identical layers and parameter counts; only the nonlinearity differs, isolating it as the cause &mdash; the paper's §3.1 claim reproduced on toy data._

**Answer:** The ReLU net's training loss falls faster in the first epochs while the tanh net lags. Since
                 the two nets are identical except for the activation, this isolates the nonlinearity: ReLU's
                 non-saturating, gradient-$1$ positive region lets learning proceed where tanh's saturation
                 stalls it. This reproduces the paper's "ReLU trains several times faster" result (§3.1, Fig. 1)
                 on a small CIFAR-10 run &mdash; our numbers, not the paper's.

</details>

**Problem 2.** Apply the conv-output-size formula. A $32\times32$ CIFAR image enters a conv layer with $5\times5$
            kernels, stride $1$, padding $2$. What is the output spatial size, and why did the designer pick
            padding $2$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug in: $O = \lfloor (32 - 5 + 2\cdot 2)/1 \rfloor + 1 = \lfloor 31/1 \rfloor + 1 = 31 + 1 = 32$. — _$W=32$, $K=5$, $P=2$, $S=1$ in $O=\lfloor (W-K+2P)/S \rfloor + 1$._
- Note the output is $32\times32$ &mdash; the same as the input. — _Padding of $(K-1)/2 = 2$ with stride $1$ keeps the spatial size unchanged ("same" convolution)._
- Conclude the designer used $P=2$ to preserve spatial size so the depth of the stack is not limited by shrinking too fast. — _On small $32\times32$ inputs, repeatedly shrinking would leave nothing to convolve; "same" padding lets you stack more conv layers._

**Answer:** $O = \lfloor (32 - 5 + 4)/1 \rfloor + 1 = 32$ &mdash; the output is $32\times32$, the same
                 size as the input. Padding $2 = (K-1)/2$ with stride $1$ is "same" convolution: it preserves the
                 spatial size so you can stack many conv layers on a small image without the feature map vanishing.

</details>

**Problem 3.** At test time you forget to call net.eval(), so dropout stays active. You evaluate the
            same test image twice and get two different predictions. Explain why, and what the fix is.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what dropout does: during training it randomly zeros each FC neuron with probability $p=0.5$, independently each forward pass. — _Dropout injects randomness so the net cannot rely on any single neuron (§4.2)._
- Realize that with dropout still on at test time, each forward pass zeros a different random half of the neurons, so the output changes run to run. — _The randomness is meant for training only; leaving it on makes inference non-deterministic and weaker._
- Fix: call net.eval() before testing (and net.train() before training). — _eval() switches dropout off and uses the full network deterministically &mdash; the intended test-time behaviour._

**Answer:** Dropout is a training-time regularizer: it randomly zeros half the FC neurons on each
                 forward pass. Left active at test time, every pass drops a different random subset, so the same
                 image yields different predictions. The fix is net.eval(), which turns dropout off
                 and runs the full deterministic network at inference (and net.train() to turn it
                 back on for training).

</details>